# Lightweight Single-Image Super-Resolution with IMDN-Lite
### Deep Learning Project


---

This notebook implements a lightweight single-image super-resolution (SISR) system targeting ×2 upscaling on CPU-constrained hardware. The core model is IMDN-lite, a scaled-down variant of the Information Multi-Distillation Network (Hui et al., 2019). The notebook is organized as follows:

| Section | Description |
|---------|-------------|
| 0 | Environment setup: Google Drive mount and library imports |
| 1 | Reproducibility and configuration |
| 2 | Dataset merging, train/validation split, and data pipeline |
| 3 | Evaluation metrics: PSNR and SSIM |
| 4 | Bicubic interpolation baseline |
| 5 | IMDN-lite architecture |
| 6 | Training loop with step-based scheduling |
| 7 | Results, training curves, and visual comparison |

**Dataset overview.** A teammate prepared two subsets of DIV2K (Agustsson and Timofte, 2017) with scale factor ×2 and a fixed random seed of 42:

| Subset | HR images | LR images | Location |
|--------|-----------|-----------|----------|
| Original | 100 | 100 | `div2k_subset/HR` and `div2k_subset/LR_x2` |
| Extended | 100 | 100 | `new_samples/div2k_subset/HR` and `new_samples/div2k_subset/LR_x2` |

Both subsets are merged into a single local directory before training, giving 200 images in total. The 90/10 train/validation split is applied after merging, with seed=42 to ensure reproducibility.


## Section 0: Environment Setup

### 0.1 Library Imports

All third-party libraries used throughout the notebook are imported here.
PyTorch is used for model definition and training. scikit-image provides the
reference SSIM implementation. All computation runs on CPU, as required by
the project specification.

In [ ]:
import os
import time
import random
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from skimage.metrics import structural_similarity as sk_ssim

print(f"PyTorch version : {torch.__version__}")
print("Device          : CPU (GPU not assumed -- project requirement)")
print("All imports loaded successfully.")


## Section 1: Reproducibility and Configuration

All hyperparameters are collected into a single `CONFIG` dictionary and saved
to disk immediately, so any run can be reproduced exactly from the saved JSON.
The random seed is set globally before any stochastic operation.

### 1.1 Seed Initialization

In [ ]:
def set_seed(seed: int = 42) -> None:
    """Fix all random-number generators for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"Seed fixed to {seed}.")

set_seed(42)


### 1.2 Configuration

The table below summarises the key design choices and the rationale for each.

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `scale` | 2 | Standard ×2 SR benchmark |
| `hr_patch_size` | 128 | Larger patches than Run 1 (96) improve texture learning |
| `batch_size` | 16 | Fits comfortably in CPU RAM |
| `max_steps` | 1000 | Step-based budget avoids epoch-length artefacts |
| `num_features` | 32 | Lightweight: paper uses 64; halved for CPU deployment |
| `num_blocks` | 3 | Paper uses 6; halved for the same reason |
| `learning_rate` | 2e-4 | Adam default for SR tasks (Hui et al., 2019) |
| `lr_decay_epochs` | 40 | StepLR milestone in pseudo-epoch units |
| `lr_decay_factor` | 0.5 | Halve LR at each milestone |
| `grad_clip` | 1.0 | Prevents gradient explosion on small batches |


In [ ]:
CONFIG = {
    # Data
    "scale"           : 2,
    "hr_patch_size"   : 128,   # HR crop; LR crop = hr_patch_size / scale
    "lr_patch_size"   : 64,
    "val_fraction"    : 0.10,

    # Model
    "num_features"    : 32,
    "num_blocks"      : 3,

    # Training
    "batch_size"      : 16,
    "max_steps"       : 1000,
    "learning_rate"   : 2e-4,
    "lr_decay_epochs" : 40,
    "lr_decay_factor" : 0.5,
    "grad_clip"       : 1.0,

    # Output
    "seed"            : 42,
    "checkpoint_dir"  : "checkpoints",
    "results_dir"     : "results",
    "save_every"      : 10,
    "log_every"       : 5,
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["results_dir"],    exist_ok=True)

# Persist config for reproducibility
with open(f"{CONFIG['results_dir']}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Configuration saved to results/config.json.")
print(json.dumps(CONFIG, indent=2))


## Section 2: Data Pipeline

### Overview

The dataset is built from two DIV2K subsets prepared by the teammate. Before
any splitting, the two subsets are merged into a single local directory so
that all file-access goes through the faster local SSD rather than Google
Drive's network mount.

The train/validation split is performed after merging. Using `seed=42` and a
90/10 ratio reproduces the teammate's original split on the 100-image subset
and extends it consistently to the full 200-image set.

Two `Dataset` classes are defined:

| Class | Purpose | Output shape |
|-------|---------|--------------|
| `TrainDataset` | Random-crop patches for training | LR `[3, 64, 64]`, HR `[3, 128, 128]` |
| `ValDataset` | Full images for validation | LR `[3, H/2, W/2]`, HR `[3, H, W]` |

Training on patches is standard in SR literature (Lim et al., 2017) because
it allows many more updates per epoch and provides implicit data augmentation
through the random crop location.


### 2.1 Dataset Merging

In [ ]:
# Data paths
BASE1 = Path(r'C:\Users\DPQUAI250142\Desktop\DL_project\patch_dataset\patch_dataset')
BASE2 = Path(r'C:\Users\DPQUAI250142\Desktop\DL_project\patches_Dataset')
CONFIG = {#  Data
          'scale'  : 2,

         #  Model
         'num_features': 32,
         'num_blocks'  : 3,

        # Training
       'batch_size'      : 32,
       "num_epochs"      : 30,
       'learning_rate'   : 2e-4,
       'lr_decay_epochs' : 40,
       'lr_decay_factor' : 0.5,
       'grad_clip'       : 1.0,

       # Output
       'seed': 42,
       'checkpoint_dir': 'checkpoints',
       'results_dir': 'results',
       'save_every': 100,
       'log_every': 50}

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['results_dir'],    exist_ok=True)

with open(f"{CONFIG['results_dir']}/config.json", 'w') as f:
    json.dump(CONFIG, f, indent=2)

print('Config saved:')
print(json.dumps(CONFIG, indent=2))

### 2.2 Train/Validation Split

In [ ]:
all_ids = sorted([
    int(f.replace(".png", ""))
    for f in os.listdir(HR_DIR)
    if f.endswith(".png")
])

random.seed(42)
shuffled = all_ids.copy()
random.shuffle(shuffled)

n_val     = int(len(shuffled) * CONFIG["val_fraction"])
val_ids   = sorted(shuffled[:n_val])
train_ids = sorted(shuffled[n_val:])

print(f"Total images : {len(all_ids)}")
print(f"Train        : {len(train_ids)}")
print(f"Validation   : {len(val_ids)}  -> IDs: {val_ids}")


### 2.3 Dataset Classes

In [ ]:
# TRAINING DATASET

class TrainDataset(Dataset):
    """
    Loads paired LR/HR PNG images.
    Extracts random aligned patches on-the-fly.

    Each call to __getitem__:
      1. Opens one full HR + LR image from disk
      2. Picks a random location in LR space
      3. Cuts LR patch (48x48) and aligned HR patch (96x96)
      4. Optional horizontal flip augmentation
      5. Returns tensors [3,48,48] and [3,96,96]
    """
    def __init__(self, hr_dir, lr_dir, image_ids,
                 scale=2, patch_size=96, augment=True):
        self.hr_paths  = [hr_dir / f"{i:04d}.png"   for i in image_ids]
        self.lr_paths  = [lr_dir / f"{i:04d}x2.png" for i in image_ids]
        self.scale      = scale
        self.patch_size = patch_size   # HR patch size
        self.augment    = augment
        self.to_tensor  = T.ToTensor()

    def __len__(self):
        return len(self.hr_paths)

    def __getitem__(self, idx):
        hr = Image.open(self.hr_paths[idx]).convert("RGB")
        lr = Image.open(self.lr_paths[idx]).convert("RGB")

        # LR patch size = HR patch size / scale = 48
        lr_ps  = self.patch_size // self.scale
        lr_w, lr_h = lr.size

        # Random top-left corner in LR space
        x = random.randint(0, lr_w - lr_ps)
        y = random.randint(0, lr_h - lr_ps)

        # Aligned crop: same spatial region in both LR and HR
        lr_patch = lr.crop((x,                   y,
                            x + lr_ps,            y + lr_ps))
        hr_patch = hr.crop((x * self.scale,       y * self.scale,
                            (x + lr_ps) * self.scale,
                            (y + lr_ps) * self.scale))

        # Horizontal flip augmentation (same as paper)
        if self.augment and random.random() > 0.5:
            lr_patch = lr_patch.transpose(Image.FLIP_LEFT_RIGHT)
            hr_patch = hr_patch.transpose(Image.FLIP_LEFT_RIGHT)

        # ToTensor: PIL (H,W,C) uint8 [0,255] -> Tensor (C,H,W) float32 [0,1]
        return self.to_tensor(lr_patch), self.to_tensor(hr_patch)


# VALIDATION DATASET
# Loads FULL images - no patching
# PSNR/SSIM on full image is the standard for SR papers

class ValDataset(Dataset):
    """
    Loads full LR/HR image pairs for validation.
    No cropping - evaluates on the complete image.
    """
    def __init__(self, hr_dir, lr_dir, image_ids):
        self.hr_paths  = [hr_dir / f"{i:04d}.png"   for i in image_ids]
        self.lr_paths  = [lr_dir / f"{i:04d}x2.png" for i in image_ids]
        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.hr_paths)

    def __getitem__(self, idx):
        hr    = Image.open(self.hr_paths[idx]).convert("RGB")
        lr    = Image.open(self.lr_paths[idx]).convert("RGB")
        fname = self.hr_paths[idx].stem
        return self.to_tensor(lr), self.to_tensor(hr), fname

print(" Dataset classes defined")

### 2.4 DataLoaders and Verification

In [ ]:
# CREATE DATALOADERS

train_dataset = TrainDataset(HR_DIR, LR_DIR, train_ids,
                             patch_size=CONFIG["hr_patch_size"], augment=True)
val_dataset   = ValDataset(HR_DIR,   LR_DIR, val_ids)

train_loader = DataLoader(
    train_dataset,
    batch_size  = CONFIG["batch_size"],
    shuffle     = True,
    num_workers = 0,     # 0 = no multiprocessing, most stable on Colab
    drop_last   = True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size  = 1,     # full images: one at a time
    shuffle     = False,
    num_workers = 0,
)

#  Verify data
lr_b, hr_b       = next(iter(train_loader))
lr_v, hr_v, fn_v = next(iter(val_loader))

print("=" * 50)
print("DATA VERIFICATION")
print("=" * 50)
print(f"Train images   : {len(train_dataset)}")
print(f"Val images     : {len(val_dataset)}")
print(f"Train batches  : {len(train_loader)}")
print(f"LR patch shape : {tuple(lr_b.shape)}")
print(f"HR patch shape : {tuple(hr_b.shape)}")
print(f"Val LR shape   : {tuple(lr_v.shape)}")
print(f"Val HR shape   : {tuple(hr_v.shape)}")
print(f"Value range    : [{lr_b.min():.2f}, {lr_b.max():.2f}]")
assert hr_b.shape[-1] == lr_b.shape[-1] * CONFIG["scale"]
print("=" * 50)
print(" Data pipeline OK")

#  Visualize sample patches
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0, i].imshow(lr_b[i].permute(1,2,0).numpy())
    axes[0, i].set_title(f"LR {tuple(lr_b[i].shape)}", fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(hr_b[i].permute(1,2,0).numpy())
    axes[1, i].set_title(f"HR {tuple(hr_b[i].shape)}", fontsize=9)
    axes[1, i].axis("off")
axes[0,0].set_ylabel(f"LR ({CONFIG['lr_patch_size']}x{CONFIG['lr_patch_size']})", fontsize=11)
axes[1,0].set_ylabel(f"HR ({CONFIG['hr_patch_size']}x{CONFIG['hr_patch_size']})", fontsize=11)
plt.suptitle("Sample Training Patches", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/sample_patches.png",
            dpi=150, bbox_inches="tight")
plt.show()

## Section 3: Evaluation Metrics

Two standard metrics are used throughout:

**PSNR (Peak Signal-to-Noise Ratio)** measures pixel-level fidelity:

$$\text{PSNR} = 10 \cdot \log_{10}\!\left(\frac{1}{\text{MSE}}\right) \quad [\text{dB}]$$

Higher values indicate closer agreement with the ground truth. For ×2 SR on
DIV2K, published IMDN results are approximately 34.6 dB (Hui et al., 2019).

**SSIM (Structural Similarity Index)** measures perceptual similarity in terms
of luminance, contrast, and structure. It lies in [0, 1], with 1 indicating
a perfect match.

Both metrics are computed on the Y (luminance) channel of YCbCr, following
the standard practice in SR papers. This avoids penalising colour shifts that
are perceptually less important than luminance errors.


In [ ]:
def rgb_to_y(img: torch.Tensor) -> torch.Tensor:
    """
    Convert an RGB tensor to the Y (luminance) channel of YCbCr.

    Follows the BT.601 coefficients used in SR benchmarks.

    Args:
        img: Float tensor with values in [0, 1].
             Shape [C, H, W] or [B, C, H, W].
    Returns:
        Y channel tensor of shape [1, H, W] or [B, 1, H, W].
    """
    squeeze = img.dim() == 3
    if squeeze:
        img = img.unsqueeze(0)
    r, g, b = img[:, 0:1], img[:, 1:2], img[:, 2:3]
    y = 16 / 255 + (65.481 / 255) * r + (128.553 / 255) * g + (24.966 / 255) * b
    return y.squeeze(0) if squeeze else y


def compute_psnr(sr: torch.Tensor, hr: torch.Tensor, use_y: bool = True) -> float:
    """
    Compute PSNR in dB between a super-resolved image and its reference.

    Args:
        sr: Super-resolved tensor, same shape as hr.
        hr: Ground-truth high-resolution tensor.
        use_y: If True, convert to Y channel before computing.
    Returns:
        PSNR value in dB.
    """
    with torch.no_grad():
        if use_y:
            sr, hr = rgb_to_y(sr.float()), rgb_to_y(hr.float())
        mse = F.mse_loss(sr.float(), hr.float())
        if mse == 0:
            return float('inf')
        return (10.0 * torch.log10(torch.tensor(1.0) / mse)).item()


def compute_ssim(sr: torch.Tensor, hr: torch.Tensor, use_y: bool = True) -> float:
    """
    Compute SSIM between a super-resolved image and its reference.

    Args:
        sr: Super-resolved tensor of shape [C, H, W].
        hr: Ground-truth tensor of shape [C, H, W].
        use_y: If True, convert to Y channel before computing.
    Returns:
        SSIM value in [0, 1].
    """
    if sr.dim() == 4:
        sr, hr = sr.squeeze(0), hr.squeeze(0)
    if use_y:
        sr_np = rgb_to_y(sr).squeeze(0).numpy()
        hr_np = rgb_to_y(hr).squeeze(0).numpy()
        ch_ax = None
    else:
        sr_np = sr.permute(1, 2, 0).numpy().clip(0, 1)
        hr_np = hr.permute(1, 2, 0).numpy().clip(0, 1)
        ch_ax = 2
    return float(sk_ssim(hr_np, sr_np, data_range=1.0, channel_axis=ch_ax, win_size=11))


# Sanity check: identical tensors should yield infinite PSNR and SSIM = 1.
t1 = torch.rand(1, 3, 96, 96)
t2 = torch.rand(1, 3, 96, 96)
print(f"Identical inputs  -> PSNR: {compute_psnr(t1, t1):.1f} dB  "
      f"SSIM: {compute_ssim(t1, t1):.4f}")
print(f"Independent noise -> PSNR: {compute_psnr(t1, t2):.2f} dB  "
      f"SSIM: {compute_ssim(t1, t2):.4f}")
print("Metric functions OK.")


## Section 4: Bicubic Interpolation Baseline

Bicubic interpolation is the standard classical baseline for SR. It has zero
learnable parameters and requires no training. Each output pixel is computed
as a weighted average of the 16 nearest input pixels using a cubic kernel.

This baseline sets the performance floor that the learned model must exceed.
Any IMDN-lite result below the bicubic PSNR would indicate a failure of
training rather than a genuine limitation of the architecture.


In [ ]:
def bicubic_upscale(lr: torch.Tensor, scale: int) -> torch.Tensor:
    """
    Upscale a low-resolution tensor using bicubic interpolation.

    Args:
        lr: LR tensor of shape [C, H, W] or [B, C, H, W].
        scale: Integer upscaling factor.
    Returns:
        Upscaled tensor, same number of dimensions as input.
    """
    squeeze = lr.dim() == 3
    if squeeze:
        lr = lr.unsqueeze(0)
    sr = F.interpolate(
        lr.float(), scale_factor=scale,
        mode='bicubic', align_corners=False, antialias=True
    ).clamp(0.0, 1.0)
    return sr.squeeze(0) if squeeze else sr


def evaluate_bicubic(val_loader, scale: int) -> dict:
    psnr_list, ssim_list, time_list = [], [], []
    for lr, hr, _ in val_loader:
        t0 = time.time()
        sr = bicubic_upscale(lr, scale)
        time_list.append((time.time() - t0) * 1000)
        psnr_list.append(compute_psnr(sr, hr, use_y=True))
        ssim_list.append(compute_ssim(sr.squeeze(0), hr.squeeze(0), use_y=True))
    return {
        "psnr"        : float(np.mean(psnr_list)),
        "ssim"        : float(np.mean(ssim_list)),
        "psnr_std"    : float(np.std(psnr_list)),
        "avg_time_ms" : float(np.mean(time_list)),
    }


print("Evaluating bicubic baseline on the validation set ...")
bicubic_metrics = evaluate_bicubic(val_loader, CONFIG["scale"])

print("\n" + "=" * 52)
print("Bicubic baseline -- full validation images (Y-channel)")
print("=" * 52)
print(f"  PSNR : {bicubic_metrics['psnr']:.4f} dB  "
      f"(+/- {bicubic_metrics['psnr_std']:.4f})")
print(f"  SSIM : {bicubic_metrics['ssim']:.4f}")
print(f"  Time : {bicubic_metrics['avg_time_ms']:.2f} ms per image")
print("=" * 52)
print("  The learned model must exceed this value.")


## Section 5: IMDN-Lite Architecture

The architecture follows the Information Multi-Distillation Network (Hui et al.,
2019) with scaled-down channel width and depth to meet the CPU inference budget.
Three modules are defined in bottom-up order.

### Architecture overview

```
Input LR image  [3, H, W]
       |
    HEAD: Conv(3x3) -> [F, H, W]         F = num_features = 32
       |
    BODY: N x IMDBBlock                   N = num_blocks = 3
    + IIC: concat all intermediate outputs -> Conv(1x1)
       |
    TAIL: Conv(3x3) -> PixelShuffle(s)   s = scale = 2
       |
Output SR image [3, sH, sW]
```

### 5.1 Contrast-Aware Channel Attention (CCA)

The CCA layer is a variant of Squeeze-and-Excitation
(Hu et al., 2018) that uses the sum of channel standard deviation and mean
rather than mean alone. Standard deviation captures edge and texture energy,
which is particularly valuable for SR where high-frequency detail is the
primary objective.

The per-channel descriptor for channel c is:

$$z_c = \text{std}(x_c) + \text{mean}(x_c)$$

A two-layer pointwise network then produces a channel attention weight in [0, 1].


In [ ]:
class CCALayer(nn.Module):
    """
    Contrast-Aware Channel Attention (Hui et al., 2019, Section 3.2.2).

    Unlike standard SE blocks (Hu et al., 2018) that pool only the mean,
    CCA pools both the standard deviation and the mean. This enriches the
    channel descriptor with texture/edge information, which is critical for
    super-resolution quality.

    Args:
        num_features: Number of input/output feature channels.
        reduction: Channel reduction ratio for the bottleneck MLP.
    """

    def __init__(self, num_features: int, reduction: int = 4):
        super().__init__()
        mid = max(num_features // reduction, 4)
        self.fc = nn.Sequential(
            nn.Conv2d(num_features, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, num_features, 1, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Equation 5 from Hui et al. (2019): z_c = std(x_c) + mean(x_c)
        contrast = (x.std(dim=[2, 3], keepdim=True)
                    + x.mean(dim=[2, 3], keepdim=True))
        return x * self.fc(contrast)


### 5.2 Information Multi-Distillation Block (IMDB)

Each IMDB block implements a Progressive Refinement Module
(PRM). At each of the four convolution steps, the feature map is split:
a refined portion (1/4 of channels) is extracted and held aside, while the
remaining coarser features continue to the next step. After four steps the
retained portions are concatenated and recombined through the CCA layer and a
1x1 fusion convolution.

For our 32-channel model the split is 8 channels retained and 24 passed forward
at each step, giving 8+8+8+8 = 32 channels at the concatenation point.

A residual connection is added around the entire block, following He et al.
(2016).


In [ ]:
class IMDBBlock(nn.Module):
    """
    Information Multi-Distillation Block (Hui et al., 2019, Section 3.2).

    The block applies four convolutional steps. After each step a fraction
    of the channels (keep = num_features // 4) is extracted as a refined
    representation and the remainder continues to the next step. The four
    refined portions are concatenated, attended by CCA, fused with a 1x1
    convolution, and added to the block input via a residual connection.

    Args:
        num_features: Number of feature channels. Must be divisible by 4.
    """

    def __init__(self, num_features: int = 32):
        super().__init__()
        self.keep = num_features // 4       # channels extracted per step
        self.rest = num_features - self.keep  # channels passed to next step

        def conv_lrelu(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.LeakyReLU(0.05, inplace=True),
            )

        # Four PRM convolutions (Table 1, Hui et al., 2019, scaled to 32ch).
        self.prm1   = conv_lrelu(num_features, num_features)
        self.prm2   = conv_lrelu(self.rest,    num_features)
        self.prm3   = conv_lrelu(self.rest,    num_features)
        self.prm4   = conv_lrelu(self.rest,    self.keep)

        self.cca    = CCALayer(num_features)
        self.fusion = nn.Conv2d(num_features, num_features, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x

        out1 = self.prm1(x)
        r1, c1 = torch.split(out1, [self.keep, self.rest], dim=1)

        out2 = self.prm2(c1)
        r2, c2 = torch.split(out2, [self.keep, self.rest], dim=1)

        out3 = self.prm3(c2)
        r3, c3 = torch.split(out3, [self.keep, self.rest], dim=1)

        r4 = self.prm4(c3)

        # Equation 4: concatenate all retained portions.
        distilled = torch.cat([r1, r2, r3, r4], dim=1)  # [B, F, H, W]

        out = self.fusion(self.cca(distilled))
        return out + residual


### 5.3 Full IMDN-Lite Network

The full network has three stages:

- **HEAD**: A single 3x3 convolution projects the RGB input into F feature maps.
- **BODY**: N IMDB blocks process the features sequentially. The Iterative
  Information Collection (IIC) mechanism concatenates the HEAD output with all
  N block outputs and fuses them with a 1x1 convolution, giving the network a
  direct gradient path from the loss to the shallowest features.
- **TAIL**: A 3x3 convolution produces 3s^2 channels, which are rearranged
  into the upscaled RGB output by PixelShuffle (Shi et al., 2016).

The network is fully convolutional and therefore processes any spatial size
at inference time, including full validation images that are much larger than
the training patches.


In [ ]:
class IMDN(nn.Module):
    """
    IMDN-Lite: lightweight single-image super-resolution network.

    Adapted from Hui et al. (2019) with reduced channel width (32 vs 64)
    and block count (3 vs 6) to meet CPU inference constraints.

    Args:
        num_features: Number of internal feature channels.
        num_blocks: Number of IMDB blocks in the body.
        scale: Spatial upscaling factor.
    """

    def __init__(self, num_features: int = 32,
                 num_blocks: int = 3, scale: int = 2):
        super().__init__()
        self.scale = scale

        # HEAD: project 3-channel RGB input to F feature maps.
        self.head = nn.Conv2d(3, num_features, 3, padding=1)

        # BODY: stack of N IMDB blocks.
        self.body = nn.ModuleList([
            IMDBBlock(num_features) for _ in range(num_blocks)
        ])

        # IIC: fuse head output with all block outputs via 1x1 convolution.
        # Input channels = (num_blocks + 1) * num_features.
        self.iic     = nn.Conv2d((num_blocks + 1) * num_features, num_features, 1)
        self.lr_conv = nn.Conv2d(num_features, num_features, 3, padding=1)

        # TAIL: sub-pixel convolution for upsampling.
        self.tail = nn.Sequential(
            nn.Conv2d(num_features, 3 * (scale ** 2), 3, padding=1),
            nn.PixelShuffle(scale),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        head_out   = self.head(x)
        iic_inputs = [head_out]
        current    = head_out

        for block in self.body:
            current = block(current)
            iic_inputs.append(current)

        # IIC fusion with global residual connection.
        fused = self.lr_conv(self.iic(torch.cat(iic_inputs, dim=1))) + head_out

        return self.tail(fused).clamp(0.0, 1.0)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("IMDN-Lite architecture defined.")


### 5.4 Model Instantiation and Shape Verification

In [ ]:
set_seed(CONFIG["seed"])

model = IMDN(
    num_features = CONFIG["num_features"],
    num_blocks   = CONFIG["num_blocks"],
    scale        = CONFIG["scale"],
)

print("=" * 56)
print("Model summary")
print("=" * 56)
print(f"  num_features   : {CONFIG['num_features']}  (original paper: 64)")
print(f"  num_blocks     : {CONFIG['num_blocks']}    (original paper: 6)")
print(f"  scale          : x{CONFIG['scale']}")
print(f"  Trainable parameters : {model.count_parameters():,}")
print("=" * 56)

# Verify output shape on a training patch.
dummy_patch = torch.zeros(1, 3, CONFIG["lr_patch_size"], CONFIG["lr_patch_size"])
with torch.no_grad():
    out_patch = model(dummy_patch)
expected_patch = (1, 3,
                  CONFIG["lr_patch_size"] * CONFIG["scale"],
                  CONFIG["lr_patch_size"] * CONFIG["scale"])
assert tuple(out_patch.shape) == expected_patch, \
    f"Shape mismatch: got {tuple(out_patch.shape)}, expected {expected_patch}."
print(f"\nPatch forward pass  : {tuple(dummy_patch.shape)} -> {tuple(out_patch.shape)}")

# Verify output shape on a full validation image.
dummy_full = torch.zeros(1, 3, 480, 640)
with torch.no_grad():
    out_full = model(dummy_full)
assert out_full.shape == (1, 3, 960, 1280)
print(f"Full image pass     : {tuple(dummy_full.shape)} -> {tuple(out_full.shape)}")
print("\nModel is fully convolutional and works at any spatial resolution.")


## Section 6: Training

### Training strategy

Training follows the step-based schedule used in lightweight SR literature
(Ahn et al., 2018; Hui et al., 2019):

- **Loss function**: L1 (mean absolute error), which produces sharper outputs
  than L2/MSE by not over-penalising large errors (Zhao et al., 2017).
- **Optimizer**: Adam with beta1=0.9, beta2=0.999 (Kingma and Ba, 2015).
- **Learning rate schedule**: StepLR with decay factor 0.5, applied every
  `lr_decay_epochs` pseudo-epochs.
- **Gradient clipping**: norm clipped to 1.0 to stabilise training.
- **Validation frequency**: every 50 steps, on a central 256x256 crop of
  each validation image. Using a fixed crop rather than the full image makes
  each validation call fast enough to run frequently without dominating
  training time on CPU.

The best checkpoint (highest validation PSNR) is saved separately and loaded
for the final evaluation in Section 7.


### 6.1 Utility Functions

In [ ]:
def validate_model(model, val_loader, scale: int) -> dict:
    """
    Evaluate the model on the central 256x256 crop of each validation image.

    Cropping is applied consistently across all epochs so that results at
    different checkpoints are directly comparable. Full-image evaluation is
    performed separately in Section 7.

    Args:
        model: The IMDN-lite model in eval mode.
        val_loader: DataLoader yielding (lr, hr, fname) tuples.
        scale: Spatial upscaling factor.
    Returns:
        Dictionary with mean PSNR, SSIM, their standard deviation, and
        average inference time in milliseconds.
    """
    model.eval()
    psnr_list, ssim_list, time_list = [], [], []
    EVAL_CROP = 256

    with torch.no_grad():
        for lr, hr, _ in val_loader:
            _, _, H, W = hr.shape
            cy = (H - EVAL_CROP) // 2
            cx = (W - EVAL_CROP) // 2

            hr_crop = hr[:, :, cy:cy + EVAL_CROP, cx:cx + EVAL_CROP]
            lr_crop = lr[:, :,
                         cy // scale:(cy + EVAL_CROP) // scale,
                         cx // scale:(cx + EVAL_CROP) // scale]

            t0 = time.time()
            sr = model(lr_crop)
            time_list.append((time.time() - t0) * 1000)

            psnr_list.append(compute_psnr(sr, hr_crop, use_y=True))
            ssim_list.append(compute_ssim(sr.squeeze(0), hr_crop.squeeze(0), use_y=True))

    return {
        "psnr"        : float(np.mean(psnr_list)),
        "ssim"        : float(np.mean(ssim_list)),
        "psnr_std"    : float(np.std(psnr_list)),
        "avg_time_ms" : float(np.mean(time_list)),
    }


def save_checkpoint(model, optimizer, scheduler, step, metrics, config,
                    is_best: bool = False) -> None:
    state = {
        "step"            : step,
        "model_state"     : model.state_dict(),
        "optimizer_state" : optimizer.state_dict(),
        "scheduler_state" : scheduler.state_dict(),
        "metrics"         : metrics,
        "config"          : config,
    }
    torch.save(state, f"{config['checkpoint_dir']}/checkpoint_step{step:04d}.pth")
    if is_best:
        torch.save(state, f"{config['checkpoint_dir']}/best_model.pth")


def load_checkpoint(path, model, optimizer=None, scheduler=None) -> dict:
    state = torch.load(path, map_location="cpu")
    model.load_state_dict(state["model_state"])
    if optimizer:  optimizer.load_state_dict(state["optimizer_state"])
    if scheduler:  scheduler.load_state_dict(state["scheduler_state"])
    print(f"Checkpoint loaded from step {state['step']}.")
    return state


print("Training utility functions defined.")


### 6.2 Optimizer and Scheduler

In [ ]:
set_seed(CONFIG["seed"])

model = IMDN(
    num_features = CONFIG["num_features"],
    num_blocks   = CONFIG["num_blocks"],
    scale        = CONFIG["scale"],
)

# L1 loss avoids the blurry-output problem associated with L2/MSE (Equation 2,
# Hui et al., 2019).
criterion = nn.L1Loss()

# Adam optimizer with the hyperparameters from the original paper.
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"], betas=(0.9, 0.999)
)

# StepLR decays the learning rate by gamma every step_size pseudo-epochs.
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size = CONFIG["lr_decay_epochs"],
    gamma     = CONFIG["lr_decay_factor"],
)

print(f"Loss function : L1 (MAE)")
print(f"Optimizer     : Adam, lr={CONFIG['learning_rate']}, beta=(0.9, 0.999)")
print(f"Scheduler     : StepLR, step_size={CONFIG['lr_decay_epochs']}, "
      f"gamma={CONFIG['lr_decay_factor']}")
print(f"Parameters    : {model.count_parameters():,}")


### 6.3 Main Training Loop

In [ ]:
set_seed(42)

history = {
    "train_loss" : [],
    "train_psnr" : [],
    "val_psnr"   : [],
    "val_ssim"   : [],
    "lr"         : [],
}
best_psnr, best_step = 0.0, 0
start_time = time.time()

MAX_STEPS      = CONFIG["max_steps"]
VALIDATE_EVERY = 50   # validation and logging frequency (in steps)

print("=" * 78)
print(f"Training IMDN-Lite | {MAX_STEPS} steps | {len(train_dataset)} images")
print(f"Bicubic baseline PSNR = {bicubic_metrics['psnr']:.4f} dB  (target to exceed)")
print("=" * 78)

global_step = 0
epoch       = 0

while global_step < MAX_STEPS:
    epoch += 1

    for lr_batch, hr_batch in train_loader:
        if global_step >= MAX_STEPS:
            break

        # Forward pass and loss computation.
        model.train()
        sr   = model(lr_batch)
        loss = criterion(sr, hr_batch)

        # Backward pass with gradient clipping.
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["grad_clip"])
        optimizer.step()
        global_step += 1

        # Periodic validation and checkpoint saving.
        if global_step % VALIDATE_EVERY == 0 or global_step == MAX_STEPS:
            val_m = validate_model(model, val_loader, scale=CONFIG["scale"])

            with torch.no_grad():
                train_psnr = compute_psnr(sr.detach(), hr_batch, use_y=False)

            history["train_loss"].append(loss.item())
            history["train_psnr"].append(train_psnr)
            history["val_psnr"].append(val_m["psnr"])
            history["val_ssim"].append(val_m["ssim"])
            history["lr"].append(scheduler.get_last_lr()[0])

            is_best = val_m["psnr"] > best_psnr
            if is_best:
                best_psnr = val_m["psnr"]
                best_step = global_step
                save_checkpoint(
                    model, optimizer, scheduler, global_step,
                    {"val_psnr": val_m["psnr"], "val_ssim": val_m["ssim"]},
                    CONFIG, is_best=True
                )

            elapsed = (time.time() - start_time) / 60
            marker  = "  [best]" if is_best else ""
            print(
                f"Step [{global_step:4d}/{MAX_STEPS}]  "
                f"Loss: {loss.item():.5f}  "
                f"Train PSNR: {train_psnr:5.2f}  "
                f"Val PSNR: {val_m['psnr']:6.3f} dB  "
                f"Val SSIM: {val_m['ssim']:.4f}  "
                f"{elapsed:.1f} min"
                f"{marker}"
            )

        # Apply learning rate decay.
        decay_interval = CONFIG["lr_decay_epochs"] * len(train_loader)
        if decay_interval > 0 and global_step % decay_interval == 0:
            scheduler.step()


total_time = (time.time() - start_time) / 60
gain = best_psnr - bicubic_metrics['psnr']
print("\n" + "=" * 78)
print(f"Training complete in {total_time:.1f} minutes.")
print(f"  Best validation PSNR : {best_psnr:.4f} dB  (step {best_step})")
print(f"  Bicubic baseline     : {bicubic_metrics['psnr']:.4f} dB")
print(f"  Gain over baseline   : {gain:+.4f} dB  "
      f"({'exceeded' if gain > 0 else 'did not exceed'} baseline)")
print("=" * 78)


## Section 7: Results and Analysis

This section loads the best checkpoint saved during training and evaluates it
on the full validation set. Results are presented through training curves,
a summary table, per-image scores, and visual comparisons.


### 7.1 Training Curves

In [ ]:
steps = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(steps, history["train_loss"], color="steelblue", lw=2)
axes[0].set_title("Training Loss (L1)", fontweight="bold")
axes[0].set_xlabel("Validation Checkpoint (every 50 steps)")
axes[0].set_ylabel("L1 Loss")
axes[0].grid(alpha=0.3)

axes[1].plot(steps, history["train_psnr"], color="steelblue", lw=2,
             label="Train PSNR (RGB)")
axes[1].plot(steps, history["val_psnr"],   color="orange",    lw=2,
             label="Val PSNR (Y-channel)")
axes[1].axhline(bicubic_metrics["psnr"], color="red", ls="--", lw=2,
                label=f"Bicubic {bicubic_metrics['psnr']:.2f} dB")
axes[1].set_title("PSNR (dB)", fontweight="bold")
axes[1].set_xlabel("Validation Checkpoint (every 50 steps)")
axes[1].set_ylabel("PSNR (dB)")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

axes[2].plot(steps, history["val_ssim"], color="green", lw=2,
             label="Val SSIM")
axes[2].axhline(bicubic_metrics["ssim"], color="red", ls="--", lw=2,
                label=f"Bicubic {bicubic_metrics['ssim']:.4f}")
axes[2].set_title("SSIM (Y-channel)", fontweight="bold")
axes[2].set_xlabel("Validation Checkpoint (every 50 steps)")
axes[2].set_ylabel("SSIM")
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3)

plt.suptitle(
    f"Training Curves -- IMDN-Lite x{CONFIG['scale']}  "
    f"[{CONFIG['num_features']} channels, {CONFIG['num_blocks']} blocks, "
    f"{CONFIG['max_steps']} steps, 200 training images]",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/training_curves.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Training curves saved.")


### 7.2 Final Evaluation on Full Validation Images

In [ ]:
# Load the best checkpoint and re-evaluate on full validation images.
# During training, evaluation used a 256x256 central crop for speed.
# Here we evaluate on the complete image, which is the standard benchmark
# protocol.
best_state    = load_checkpoint(f"{CONFIG['checkpoint_dir']}/best_model.pth", model)
final_metrics = validate_model(model, val_loader, scale=CONFIG["scale"])

print("\n" + "=" * 66)
print("Final results -- DIV2K validation images, Y-channel metrics")
print("=" * 66)
print(f"{'Method':<24} {'PSNR (dB)':>10} {'SSIM':>8} {'Time (ms)':>12}")
print("-" * 66)
print(f"{'Bicubic (baseline)':<24} "
      f"{bicubic_metrics['psnr']:>10.4f} "
      f"{bicubic_metrics['ssim']:>8.4f} "
      f"{bicubic_metrics['avg_time_ms']:>12.2f}")
print(f"{'IMDN-Lite (ours)':<24} "
      f"{final_metrics['psnr']:>10.4f} "
      f"{final_metrics['ssim']:>8.4f} "
      f"{final_metrics['avg_time_ms']:>12.2f}")
print("-" * 66)
gain_psnr = final_metrics['psnr'] - bicubic_metrics['psnr']
gain_ssim = final_metrics['ssim'] - bicubic_metrics['ssim']
print(f"{'Improvement':<24} {gain_psnr:>+10.4f} {gain_ssim:>+8.4f}")
print("=" * 66)
print(f"\n  Trainable parameters : {model.count_parameters():,}")
print(f"  Best checkpoint step : {best_state['step']}")


### 7.3 Per-Image Results

In [ ]:
print(f"{'Image':<10} {'Bicubic PSNR':>14} {'IMDN PSNR':>12} {'Gain':>10}")
print("-" * 50)

bic_scores, imdn_scores, fnames = [], [], []
model.eval()
with torch.no_grad():
    for lr, hr, fname in val_loader:
        sr_bic  = bicubic_upscale(lr, CONFIG["scale"])
        sr_imdn = model(lr)
        bic_scores.append(compute_psnr(sr_bic,  hr, use_y=True))
        imdn_scores.append(compute_psnr(sr_imdn, hr, use_y=True))
        fnames.append(fname[0])

for name, bic, imdn in zip(fnames, bic_scores, imdn_scores):
    gain = imdn - bic
    result_marker = "  +" if gain > 0 else "  -"
    print(f"  {name:<8} {bic:>14.4f} {imdn:>12.4f} {gain:>+9.4f}{result_marker}")

print("-" * 50)
print(f"  {'Mean':<8} {np.mean(bic_scores):>14.4f} "
      f"{np.mean(imdn_scores):>12.4f} "
      f"{np.mean(imdn_scores) - np.mean(bic_scores):>+9.4f}")


### 7.4 Visual Comparison

A 192x192 central crop is shown for each validation image. Visual inspection
complements the quantitative metrics: PSNR differences of less than 0.5 dB
are sometimes difficult to distinguish by eye, while artefacts such as ringing
or over-sharpening are immediately visible.


In [ ]:
CROP   = 192
N_SHOW = min(5, len(val_dataset))
model.eval()

fig, axes = plt.subplots(N_SHOW, 3,
                          figsize=(13, 4.5 * N_SHOW),
                          gridspec_kw={"hspace": 0.35, "wspace": 0.05})

for col, title in enumerate(["Bicubic (Baseline)", "IMDN-Lite (Ours)", "Ground Truth (HR)"]):
    axes[0, col].set_title(title, fontsize=12, fontweight="bold", pad=8)

for row, (lr, hr, fname) in enumerate(list(val_loader)[:N_SHOW]):
    with torch.no_grad():
        sr_bic  = bicubic_upscale(lr, CONFIG["scale"])
        sr_imdn = model(lr)

    _, _, H, W = hr.shape
    cy, cx = (H - CROP) // 2, (W - CROP) // 2

    def crop_arr(t):
        return t[0, :, cy:cy + CROP, cx:cx + CROP].permute(1, 2, 0).numpy().clip(0, 1)

    psnr_bic  = compute_psnr(sr_bic,  hr, use_y=True)
    psnr_imdn = compute_psnr(sr_imdn, hr, use_y=True)
    ssim_imdn = compute_ssim(sr_imdn.squeeze(0), hr.squeeze(0), use_y=True)

    for col, (img, label) in enumerate([
        (crop_arr(sr_bic),  f"PSNR: {psnr_bic:.2f} dB"),
        (crop_arr(sr_imdn), f"PSNR: {psnr_imdn:.2f} dB  SSIM: {ssim_imdn:.4f}"),
        (crop_arr(hr),      "Ground Truth"),
    ]):
        axes[row, col].imshow(img, interpolation="nearest")
        axes[row, col].set_xlabel(label, fontsize=9)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

    axes[row, 0].set_ylabel(f"Image: {fname[0]}", fontsize=10)

plt.suptitle(f"Visual Comparison -- {CROP}x{CROP} central crop",
             fontsize=13, fontweight="bold", y=1.01)
plt.savefig(f"{CONFIG['results_dir']}/visual_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Visual comparison saved.")


### 7.5 Results Summary

In [ ]:
results = {
    "config" : CONFIG,
    "data"   : {
        "total_images"  : len(train_dataset) + len(val_dataset),
        "train_images"  : len(train_dataset),
        "val_images"    : len(val_dataset),
        "train_ids"     : train_ids,
        "val_ids"       : val_ids,
    },
    "model"  : {
        "name"           : "IMDN-Lite",
        "num_features"   : CONFIG["num_features"],
        "num_blocks"     : CONFIG["num_blocks"],
        "parameters"     : model.count_parameters(),
    },
    "bicubic"     : bicubic_metrics,
    "imdn"        : final_metrics,
    "improvement" : {
        "psnr_gain_db"  : final_metrics['psnr'] - bicubic_metrics['psnr'],
        "ssim_gain"     : final_metrics['ssim'] - bicubic_metrics['ssim'],
    },
    "training"    : {
        "best_step"     : best_step,
        "total_steps"   : CONFIG["max_steps"],
        "time_minutes"  : round(total_time, 2),
        "history"       : history,
    },
    "per_image"   : {
        name: {"bicubic_psnr": bic, "imdn_psnr": imdn, "gain": imdn - bic}
        for name, bic, imdn in zip(fnames, bic_scores, imdn_scores)
    },
}

with open(f"{CONFIG['results_dir']}/results_summary.json", "w") as f:
    json.dump(results, f, indent=2)

print("All results saved.\n")
print("Generated files:")
for fp in sorted(Path(CONFIG['results_dir']).iterdir()):
    print(f"  {fp.name}")
print(f"  checkpoints/best_model.pth")

print("\n" + "=" * 56)
print("Run complete")
print("=" * 56)
print(f"  Dataset          : 200 images (100 original + 100 extended)")
print(f"  Bicubic PSNR     : {bicubic_metrics['psnr']:.4f} dB")
print(f"  IMDN-Lite PSNR   : {final_metrics['psnr']:.4f} dB")
print(f"  Gain             : {final_metrics['psnr'] - bicubic_metrics['psnr']:+.4f} dB")
print(f"  Parameters       : {model.count_parameters():,}")
print(f"  Training time    : {total_time:.1f} minutes")
print("=" * 56)
